# 1. Configuração de Persistência

Acesso ao sistema de arquivos externo para garantir a integridade dos dados de entrada e a rastreabilidade dos outputs que geraremos posteriormente.

# 2. Importação de Bibliotecas Base

Carregamento das ferramentas iniciais de sistema operacional e de manipulação de arquivos estruturados que utilizaremos para o pré-processamento.

In [1]:
# Instalação das bibliotecas base e otimizadores para Fine-Tuning eficiente (Unsloth, PEFT, Transformers)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

from google.colab import drive
import os
import json

# Conexão com o sistema de arquivos persistente
drive.mount('/content/drive')

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-2njbkt43/unsloth_003307574311429e9961d8c7d97eb1fd
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-2njbkt43/unsloth_003307574311429e9961d8c7d97eb1fd
  Resolved https://github.com/unslothai/unsloth.git to commit 614cae8793d6e08fcb8d174a29cdf0bb884e741d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 113.6 MB/s eta 0:00:00
   ━━

# Preparação e Estruturação do Dataset

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


# Configuração do tamplate de prompt e formatação do dataset bruto

In [3]:
prompt_style = """ abaixo está uma instrução que descreve uma tarefa, juntamente com uma entrada que fornece contexto adicional. Escreva uma resposta que complete adequadamente o pedido.

### Instrução:
{}

### Entrada:
{}

### Resposta:
{}"""

EOS_TOKEN = tokenizer.eos_token

# Função de Formatação do Dataset

In [4]:
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = prompt_style.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# Carregando Dataset bruto de treinamento
E aplicando a função de mapeamento para estruturar os dados utilizando o prompt definido.

In [5]:
from datasets import load_dataset, concatenate_datasets

dataset_bruto = load_dataset(
    "mukulb/clustered_MEDQUAD_dataset_with_groups",
    split="train"
)

dataset_selecionado = concatenate_datasets([
    dataset_bruto.select(range(1000)),
    dataset_bruto.select([11718]),
])

dataset = dataset_selecionado.map(
    lambda exemplo: {
        "instruction": "Responda à pergunta médica com base em informações clínicas confiáveis.",
        "input": exemplo["query"],
        "output": exemplo["answers"],
    },
    remove_columns=dataset_selecionado.column_names,
)

dataset = dataset.map(formatting_prompts_func, batched=True)

print("Total de registros preparados:", len(dataset))
print("Última pergunta:", dataset[-1]["input"])

README.md:   0%|          | 0.00/2.90k [00:00<?, ?B/s]

MedQuAD_Formatted_QA.csv: reconstructing file:   0%|          |  0.00B / 45.0MB            

MedQuAD_Formatted_QA.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/16407 [00:00<?, ? examples/s]

Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

Total de registros preparados: 1001
Última pergunta: What is (are) Chest Pain ?


# Configurando Hiperparâmetros utilizando o adaptador LoRA para treinamento.

In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

Unsloth 2026.8.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


# TREINO DATASET MÉDICO

In [7]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs_medquad",
        save_strategy="no",
        report_to="none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1001 [00:00<?, ? examples/s]

In [8]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,001 | Num Epochs = 1 | Total steps = 126
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.374341
2,1.544612
3,1.504570
4,1.612696
5,1.131824
6,1.262475
7,1.081108
8,1.068434
9,1.020064
10,0.864859


# SALVANDO ADAPTADOR LoRA NO DRIVE

In [9]:
caminho_modelo_final = "/content/drive/MyDrive/TechChallenge3/adaptador_medquad_lora_final"

model.save_pretrained(caminho_modelo_final)
tokenizer.save_pretrained(caminho_modelo_final)

print(f"Adaptador final salvo em: {caminho_modelo_final}")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/TechChallenge3/adaptador_medquad_lora_final/tokenizer_config.json.


Adaptador final salvo em: /content/drive/MyDrive/TechChallenge3/adaptador_medquad_lora_final


In [11]:
FastLanguageModel.for_inference(model)

pergunta = "When should a patient with chest pain seek emergency care?"

prompt = prompt_style.format(
    "Responda à pergunta médica com base em informações clínicas confiáveis.",
    pergunta,
    "",
)

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=250,
    do_sample=False,
    repetition_penalty=1.1,
)

resposta = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print(resposta)

Both `max_new_tokens` (=250) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Key Points
                    - Chest pain or pressure is a common symptom of heart attack and other serious conditions.    - Patients who have had a heart attack are at risk for another heart attack.     - Older adults may not always recognize the signs of a heart attack.    - Women are more likely than men to delay seeking medical help when they experience symptoms of a heart attack.    - People who have had a heart attack should be seen by a doctor right away if they have any new or worsening symptoms.
                
                
                    Chest pain or pressure is a common symptom of heart attack and other serious conditions.
                    Heart attack occurs when blood flow to the heart is blocked by a blood clot in a coronary artery (the arteries that supply blood to the heart). The blockage prevents oxygen-rich blood from reaching part of the heart, causing damage to the heart muscle. This can lead to serious complications, including death. Other condition